In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from typing import List, Dict
import json

class CryptoDataFetcher:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://financialmodelingprep.com/stable"
        
    def fetch_crypto_list(self) -> List[Dict]:
        """Fetch the complete list of cryptocurrencies"""
        url = f"{self.base_url}/cryptocurrency-list?apikey={self.api_key}"
        
        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            print(f"✓ Fetched {len(data)} cryptocurrencies")
            return data
        except Exception as e:
            print(f"✗ Error fetching crypto list: {e}")
            return []
    
    def fetch_quote(self, symbol: str) -> Dict:
        """Fetch detailed quote data for a specific symbol"""
        url = f"{self.base_url}/quote?symbol={symbol}&apikey={self.api_key}"
        
        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            
            if data and len(data) > 0:
                return data[0]
            return None
        except Exception as e:
            print(f"✗ Error fetching quote for {symbol}: {e}")
            return None
    
    def fetch_historical_data(self, symbol: str, months: int = 24) -> pd.DataFrame:
        """Fetch historical OHLCV data for specified months"""
        # Calculate date range
        end_date = datetime.now()
        start_date = end_date - timedelta(days=months * 30)
        
        # Format dates for API
        from_date = start_date.strftime('%Y-%m-%d')
        to_date = end_date.strftime('%Y-%m-%d')
        
        url = f"{self.base_url}/historical-price-full/{symbol}?from={from_date}&to={to_date}&apikey={self.api_key}"
        
        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            
            if 'historical' in data:
                df = pd.DataFrame(data['historical'])
                df['symbol'] = symbol
                return df
            return pd.DataFrame()
        except Exception as e:
            print(f"✗ Error fetching historical data for {symbol}: {e}")
            return pd.DataFrame()
    
    def fetch_all_crypto_data(self, sample_size: int = None, delay: float = 0.2) -> pd.DataFrame:
        """
        Fetch comprehensive data for all cryptocurrencies
        
        Args:
            sample_size: If specified, only fetch this many cryptos (for testing)
            delay: Delay between API calls to respect rate limits
        """
        print("=" * 70)
        print("CRYPTOCURRENCY DATA COLLECTION")
        print("=" * 70)
        
        # Step 1: Get list of all cryptocurrencies
        print("\n[1/3] Fetching cryptocurrency list...")
        crypto_list = self.fetch_crypto_list()
        
        if not crypto_list:
            print("No cryptocurrencies found. Exiting.")
            return pd.DataFrame()
        
        # Limit sample size if specified
        if sample_size:
            crypto_list = crypto_list[:sample_size]
            print(f"📊 Processing sample of {sample_size} cryptocurrencies")
        
        # Step 2: Fetch quote data for each cryptocurrency
        print(f"\n[2/3] Fetching detailed quotes for {len(crypto_list)} cryptocurrencies...")
        print("(This may take a while...)")
        
        all_data = []
        failed_symbols = []
        
        for idx, crypto in enumerate(crypto_list, 1):
            symbol = crypto['symbol']
            
            # Progress indicator
            if idx % 50 == 0 or idx == len(crypto_list):
                print(f"Progress: {idx}/{len(crypto_list)} ({idx/len(crypto_list)*100:.1f}%)")
            
            # Fetch quote data
            quote_data = self.fetch_quote(symbol)
            
            if quote_data:
                # Combine list data + quote data
                combined = {**crypto, **quote_data}
                all_data.append(combined)
            else:
                failed_symbols.append(symbol)
            
            # Rate limiting
            time.sleep(delay)
        
        # Step 3: Create DataFrame
        print(f"\n[3/3] Creating dataset...")
        df = pd.DataFrame(all_data)
        
        # Add derived metrics
        if not df.empty:
            df = self._add_derived_metrics(df)
        
        # Summary
        print("\n" + "=" * 70)
        print("COLLECTION SUMMARY")
        print("=" * 70)
        print(f"✓ Successfully fetched: {len(all_data)} cryptocurrencies")
        print(f"✗ Failed to fetch: {len(failed_symbols)} cryptocurrencies")
        if failed_symbols[:5]:
            print(f"  Examples: {', '.join(failed_symbols[:5])}")
        print(f"📊 Total columns: {len(df.columns)}")
        print("=" * 70)
        
        return df
    
    def _add_derived_metrics(self, df: pd.DataFrame) -> pd.DataFrame:
        """Add derived statistical features"""
        
        # Volatility metrics (using day high/low as proxy)
        if 'dayHigh' in df.columns and 'dayLow' in df.columns and 'price' in df.columns:
            df['daily_range_pct'] = ((df['dayHigh'] - df['dayLow']) / df['price'] * 100).round(2)
        
        # Year-to-date performance
        if 'price' in df.columns and 'yearLow' in df.columns:
            df['ytd_gain_pct'] = ((df['price'] - df['yearLow']) / df['yearLow'] * 100).round(2)
        
        # Distance from averages
        if 'price' in df.columns and 'priceAvg50' in df.columns:
            df['distance_from_50ma_pct'] = ((df['price'] - df['priceAvg50']) / df['priceAvg50'] * 100).round(2)
        
        if 'price' in df.columns and 'priceAvg200' in df.columns:
            df['distance_from_200ma_pct'] = ((df['price'] - df['priceAvg200']) / df['priceAvg200'] * 100).round(2)
        
        # Market cap categories
        if 'marketCap' in df.columns:
            df['market_cap_category'] = pd.cut(
                df['marketCap'], 
                bins=[0, 1e7, 1e8, 1e9, 1e10, float('inf')],
                labels=['Micro (<10M)', 'Small (10M-100M)', 'Mid (100M-1B)', 'Large (1B-10B)', 'Mega (>10B)']
            )
        
        # Volume to market cap ratio
        if 'volume' in df.columns and 'marketCap' in df.columns:
            df['volume_to_mcap_ratio'] = (df['volume'] / df['marketCap'] * 100).round(2)
        
        # Data collection timestamp
        df['data_collected_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        return df
    
    def save_to_csv(self, df: pd.DataFrame, filename: str = None):
        """Save DataFrame to CSV"""
        if filename is None:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"crypto_data_{timestamp}.csv"
        
        filepath = f"/mnt/user-data/outputs/{filename}"
        df.to_csv(filepath, index=False)
        print(f"\n💾 Data saved to: {filename}")
        print(f"   Rows: {len(df):,}")
        print(f"   Columns: {len(df.columns)}")
        
        return filepath


def main():
    # Your API key
    API_KEY = "m8TZJWQFGH7G6x2nowAqKdzDfAyakr0T"
    
    # Initialize fetcher
    fetcher = CryptoDataFetcher(API_KEY)
    
    # Fetch data
    # For testing, you can use sample_size parameter: sample_size=100
    # For full dataset, remove sample_size or set to None
    df = fetcher.fetch_all_crypto_data(sample_size=None, delay=0.2)
    
    if not df.empty:
        # Save to CSV
        filepath = fetcher.save_to_csv(df, filename="../outputs/cryptocurrency_market_data.csv")
        
        # Display sample
        print("\n" + "=" * 70)
        print("DATA PREVIEW (First 5 rows)")
        print("=" * 70)
        print(df.head().to_string())
        
        print("\n" + "=" * 70)
        print("COLUMN LIST")
        print("=" * 70)
        for col in df.columns:
            print(f"  • {col}")
        
        # Basic statistics
        print("\n" + "=" * 70)
        print("KEY STATISTICS")
        print("=" * 70)
        if 'marketCap' in df.columns:
            print(f"Total Market Cap: ${df['marketCap'].sum():,.0f}")
        if 'volume' in df.columns:
            print(f"Total 24h Volume: ${df['volume'].sum():,.0f}")
        if 'price' in df.columns:
            print(f"Average Price: ${df['price'].mean():,.2f}")
            print(f"Median Price: ${df['price'].median():,.2f}")
        
        return filepath
    else:
        print("\n⚠️  No data collected.")
        return None


if __name__ == "__main__":
    main()

CRYPTOCURRENCY DATA COLLECTION

[1/3] Fetching cryptocurrency list...
✓ Fetched 4785 cryptocurrencies

[2/3] Fetching detailed quotes for 4785 cryptocurrencies...
(This may take a while...)
Progress: 50/4785 (1.0%)
Progress: 100/4785 (2.1%)
Progress: 150/4785 (3.1%)
Progress: 200/4785 (4.2%)
Progress: 250/4785 (5.2%)
Progress: 300/4785 (6.3%)
Progress: 350/4785 (7.3%)
Progress: 400/4785 (8.4%)
Progress: 450/4785 (9.4%)
Progress: 500/4785 (10.4%)
Progress: 550/4785 (11.5%)
Progress: 600/4785 (12.5%)
Progress: 650/4785 (13.6%)
Progress: 700/4785 (14.6%)
Progress: 750/4785 (15.7%)
Progress: 800/4785 (16.7%)
Progress: 850/4785 (17.8%)
Progress: 900/4785 (18.8%)
Progress: 950/4785 (19.9%)
Progress: 1000/4785 (20.9%)
Progress: 1050/4785 (21.9%)
Progress: 1100/4785 (23.0%)
Progress: 1150/4785 (24.0%)
Progress: 1200/4785 (25.1%)
Progress: 1250/4785 (26.1%)
Progress: 1300/4785 (27.2%)
Progress: 1350/4785 (28.2%)
Progress: 1400/4785 (29.3%)
Progress: 1450/4785 (30.3%)
✗ Error fetching quote for 

KeyboardInterrupt: 